# Data Preparation
Builds training data from a Planet NICFI mosaic and digitised polygons. Outputs a spatially-split
patch archive (`split.tar.gz`) ready for the training notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get install -y gdal-bin > /dev/null 2>&1
!pip install rasterio geopandas scikit-learn -q

In [ ]:
import os, glob, shutil, subprocess
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
import geopandas as gpd
from sklearn.model_selection import GroupShuffleSplit

print('Imports done')

In [ ]:
TILES_DIR = '/content/drive/MyDrive/Gala_datasets/images'
SHAPEFILE_PATH = '/content/drive/MyDrive/Gala_datasets/training_polygons.shp'
DRIVE_SAVE_DIR = '/content/drive/MyDrive/Gala_datasets'

WORK_DIR = '/content/work'
os.makedirs(WORK_DIR, exist_ok=True)

PATCH_SIZE = 256
OVERLAP = 64
MIN_LABELED_PCT = 5.0
BLOCK_SIZE = 1024          # spatial block size for train/val/test splitting (~4.9 km at 4.77 m/px)
STRIP_HEIGHT = 2048
RANDOM_SEED = 42

print(f'Tiles:   {TILES_DIR}')
print(f'Labels:  {SHAPEFILE_PATH}')

## Build mosaic and rasterise training polygons

In [ ]:
# Build VRT if multiple tiles, otherwise use the single file directly
tiles = sorted(glob.glob(os.path.join(TILES_DIR, '*.tif')))
print(f'Found {len(tiles)} tile(s)')

if len(tiles) == 1:
    MOSAIC_PATH = tiles[0]
    print(f'Single file: {MOSAIC_PATH}')
else:
    MOSAIC_PATH = os.path.join(WORK_DIR, 'mosaic.vrt')
    subprocess.run(['gdalbuildvrt', MOSAIC_PATH] + tiles, check=True)
    print(f'VRT built: {MOSAIC_PATH}')

with rasterio.open(MOSAIC_PATH) as src:
    print(f'Dimensions: {src.width} x {src.height}')
    print(f'Bands: {src.count}')
    print(f'CRS: {src.crs}')

In [ ]:
gdf = gpd.read_file(SHAPEFILE_PATH)
print(f'Loaded {len(gdf)} polygons')
print(f'Classes: {gdf["Class"].value_counts().to_dict()}')

In [ ]:
MASK_PATH = os.path.join(WORK_DIR, 'mask.tif')

with rasterio.open(MOSAIC_PATH) as src:
    height, width = src.height, src.width
    transform = src.transform
    crs = src.crs

if gdf.crs != crs:
    print(f'Reprojecting from {gdf.crs} to {crs}')
    gdf = gdf.to_crs(crs)

shapes_0 = [(geom, 0) for geom in gdf[gdf['Class'] == 0].geometry if geom is not None]
shapes_1 = [(geom, 1) for geom in gdf[gdf['Class'] == 1].geometry if geom is not None]
print(f'Class 0: {len(shapes_0)} | Class 1: {len(shapes_1)}')

mask_profile = {
    'driver': 'GTiff', 'dtype': 'uint8', 'width': width, 'height': height,
    'count': 1, 'crs': crs, 'transform': transform, 'compress': 'lzw',
}

# Rasterise in strips to avoid loading the full mask into memory.
# 255 = unlabelled, 0 = non-galamsey, 1 = galamsey.
# Class 1 is burned second so it overwrites Class 0 where they overlap.
with rasterio.open(MASK_PATH, 'w', **mask_profile) as dst:
    for y in range(0, height, STRIP_HEIGHT):
        h = min(STRIP_HEIGHT, height - y)
        strip_transform = rasterio.transform.from_bounds(
            transform.c,
            transform.f + (y + h) * transform.e,
            transform.c + width * transform.a,
            transform.f + y * transform.e,
            width, h
        )
        strip = np.full((h, width), 255, dtype=np.uint8)
        if shapes_0:
            rasterize(shapes_0, out=strip, transform=strip_transform, default_value=0)
        if shapes_1:
            rasterize(shapes_1, out=strip, transform=strip_transform, default_value=1)
        dst.write(strip, 1, window=Window(0, y, width, h))
        print(f'  Row {y:,} to {y+h:,}', end='\r')

print(f'\nMask saved: {MASK_PATH}')

## Patch generation and spatial splitting

In [ ]:
PATCHES_DIR = os.path.join(WORK_DIR, 'patches')
IMG_DIR = os.path.join(PATCHES_DIR, 'images')
MSK_DIR = os.path.join(PATCHES_DIR, 'masks')
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(MSK_DIR, exist_ok=True)

stride = PATCH_SIZE - OVERLAP
saved = 0
skipped = 0

with rasterio.open(MOSAIC_PATH) as img_src, rasterio.open(MASK_PATH) as msk_src:
    height, width = img_src.height, img_src.width
    img_profile = img_src.profile.copy()

    total_y = len(range(0, height - PATCH_SIZE + 1, stride))
    total_x = len(range(0, width - PATCH_SIZE + 1, stride))
    total = total_y * total_x
    print(f'Total candidate patches: {total:,}')

    for y in range(0, height - PATCH_SIZE + 1, stride):
        for x in range(0, width - PATCH_SIZE + 1, stride):
            window = Window(x, y, PATCH_SIZE, PATCH_SIZE)
            mask = msk_src.read(1, window=window)

            # Skip patches with too few labelled pixels
            labeled_pct = (mask != 255).sum() / mask.size * 100
            if labeled_pct < MIN_LABELED_PCT:
                skipped += 1
                continue

            # Skip nodata patches (all bands zero)
            image = img_src.read(window=window)
            if np.all(image == 0):
                skipped += 1
                continue

            fname = f'patch_{y:06d}_{x:06d}.tif'

            patch_profile = img_profile.copy()
            patch_profile.update(
                width=PATCH_SIZE, height=PATCH_SIZE,
                transform=rasterio.windows.transform(window, img_src.transform),
                compress='lzw',
            )
            with rasterio.open(os.path.join(IMG_DIR, fname), 'w', **patch_profile) as dst:
                dst.write(image)

            mask_profile_p = patch_profile.copy()
            mask_profile_p.update(count=1, dtype='uint8')
            with rasterio.open(os.path.join(MSK_DIR, fname.replace('.tif', '_mask.tif')), 'w', **mask_profile_p) as dst:
                dst.write(mask[np.newaxis, :, :])

            saved += 1
        print(f'  Saved: {saved:,} | Skipped: {skipped:,}', end='\r')

print(f'\nDone. Saved: {saved:,} | Skipped: {skipped:,}')

In [ ]:
# Assign each patch to a spatial block based on its pixel coordinates.
# All patches in the same block go to the same split (train/val/test)
# to prevent spatial autocorrelation from inflating metrics.
patch_files = sorted(os.listdir(IMG_DIR))
print(f'Total patches: {len(patch_files)}')

block_ids = []
for f in patch_files:
    parts = f.replace('patch_', '').replace('.tif', '').split('_')
    y, x = int(parts[0]), int(parts[1])
    block_ids.append(f'{y // BLOCK_SIZE}_{x // BLOCK_SIZE}')

block_ids = np.array(block_ids)
print(f'Spatial blocks: {len(np.unique(block_ids))}')

# Two-stage split: 70% train, then remaining 30% halved into 15% val / 15% test
splitter1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_SEED)
train_idx, valtest_idx = next(splitter1.split(patch_files, groups=block_ids))

valtest_blocks = block_ids[valtest_idx]
valtest_files = [patch_files[i] for i in valtest_idx]
splitter2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=RANDOM_SEED)
val_sub_idx, test_sub_idx = next(splitter2.split(valtest_files, groups=valtest_blocks))

val_idx = valtest_idx[val_sub_idx]
test_idx = valtest_idx[test_sub_idx]

print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')

In [ ]:
SPLIT_DIR = os.path.join(WORK_DIR, 'split')

for split_name, indices in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    img_out = os.path.join(SPLIT_DIR, split_name, 'images')
    msk_out = os.path.join(SPLIT_DIR, split_name, 'masks')
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(msk_out, exist_ok=True)

    for i in indices:
        fname = patch_files[i]
        mask_fname = fname.replace('.tif', '_mask.tif')
        shutil.copy2(os.path.join(IMG_DIR, fname), os.path.join(img_out, fname))
        shutil.copy2(os.path.join(MSK_DIR, mask_fname), os.path.join(msk_out, mask_fname))

    print(f'{split_name}: {len(indices)} patches')

## Save to Drive

In [ ]:
import tarfile

archive_path = os.path.join(WORK_DIR, 'split.tar.gz')
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(SPLIT_DIR, arcname='split')

size_mb = os.path.getsize(archive_path) / 1e6
print(f'Archive: {size_mb:.1f} MB')

drive_output = os.path.join(DRIVE_SAVE_DIR, 'split.tar.gz')
shutil.copy2(archive_path, drive_output)
print(f'Saved to: {drive_output}')